# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset
print('Available Record Sets:')
record_sets = []
for rs in metadata.recordSets:
    print(f"- @id: {rs['@id']} Name: {rs.get('name', '<No Name>')}")
    record_sets.append(rs['@id'])
    
# For each record set, list fields (columns)
for rs in metadata.recordSets:
    print(f"\nRecord Set @id: {rs['@id']} (Name: {rs.get('name', '<No Name>')})")
    print("Fields (columns):")
    for field in rs.get('field', []):
        # Each field is likely a dict with @id, name, and other properties
        if isinstance(field, dict):
            print(f"  - @id: {field.get('@id', field)} Name: {field.get('name', '<No Name>')} DataType: {field.get('dataType', '<Unknown>')}")
        else:
            print(f"  - @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load records from all record sets found above
dataframes = {}

# If no record sets found, print a message
if not record_sets:
    print('No record sets found in metadata.')
else:
    for rs_id in record_sets:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet {rs_id} (shape={df.shape})")
        except Exception as e:
            print(f"Failed to load RecordSet {rs_id}: {e}")

    # Show columns for first (or typical) record set
    if dataframes:
        displayed_rs = record_sets[0]
        print(f"\nColumns in record set {displayed_rs}:")
        print(dataframes[displayed_rs].columns.tolist())
        dataframes[displayed_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose the main record set and typical numeric and group fields based on dataset context
record_set_id = record_sets[0] if record_sets else None

if record_set_id and (record_set_id in dataframes):
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    # Try to choose a plausible numeric field (e.g., abundance, MW, etc.)
    potential_numeric_fields = ['Abundance', 'MW', 'Coverage', 'Peptides', 'UniquePeptides', 'Score', 'pI']
    found_numeric = [col for col in df.columns if any(nf.lower() in col.lower() for nf in potential_numeric_fields)]
    if found_numeric:
        numeric_field = found_numeric[0]
    else:
        # Fall back to the first column of numeric type if found
        numeric_field = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else None
    print(f"Selected numeric field: {numeric_field}")

    # Filtering: Remove NaNs and filter for values greater than threshold
    threshold = df[numeric_field].quantile(0.75) if numeric_field else None
    filtered_df = df[df[numeric_field] > threshold] if numeric_field else df
    print(f"Filtered records with {numeric_field} > {threshold} (top quartile):")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    if numeric_field:
        filtered_df = filtered_df.copy()
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

    # Group by a categorical/group field, e.g., sample group, description, etc.
    potential_group_fields = ['Group', 'Sample', 'Description', 'accession', 'Protein']
    group_field = None
    for g in potential_group_fields:
        group_candidates = [col for col in df.columns if g.lower() in col.lower()]
        if group_candidates:
            group_field = group_candidates[0]
            break
    print(f"Selected group field: {group_field}")

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No valid record set or DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and (record_set_id in dataframes) and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=25, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field and grouped_df are defined, plot group means
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,5))
        plot_df = grouped_df.sort_values(numeric_field, ascending=False).head(20)
        sns.barplot(data=plot_df, x=group_field, y=numeric_field)
        plt.title(f"Top 20 groups by mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
In this notebook, we've used `mlcroissant` to explore the FAIR² dataset of extracellular vesicle proteomics. We loaded the Croissant metadata and records, examined the available record sets and columns using their `@id`s, and performed filtering and basic normalization on the main numeric field. Further analysis may consider multi-dimensional statistics and cross-referencing with external protein databases.